# Credit Card Fraud Detection — Data Preprocessing

This notebook prepares `creditcard.csv` for model training.

**Target column:** `Class`  
**Scaling:** Only `Time` and `Amount` are scaled because `V1`–`V28` are already PCA-transformed.  
**Data leakage prevention:** The scaler is fitted on the training data only.

Keep this notebook in the same project folder as `creditcard.csv`.

## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 2. Load the dataset

In [ ]:
# The CSV file should be in the same folder as this notebook.
df = pd.read_csv("creditcard.csv")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Explore the dataset

In [ ]:
print("Dataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe())

## 4. Check missing values

In [ ]:
missing_values = df.isnull().sum()

print("Missing values by column:")
display(missing_values[missing_values > 0])

print("Total missing values:", int(missing_values.sum()))

## 5. Remove duplicate records

Duplicate rows are removed before splitting the data. The original dataset is not modified.

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

## 6. Handle missing values (if any)

For this dataset, missing values are not normally expected. If any are present, this cell fills numeric feature columns with their median. The target column must not contain missing labels.

In [ ]:
if "Class" not in df.columns:
    raise ValueError("Target column 'Class' was not found in the dataset.")

if df["Class"].isna().any():
    raise ValueError("The target column 'Class' contains missing values. Please inspect these rows.")

feature_columns = df.drop(columns=["Class"]).select_dtypes(include=np.number).columns
df[feature_columns] = df[feature_columns].fillna(df[feature_columns].median())

print("Missing values remaining:", int(df.isnull().sum().sum()))

## 7. Separate features and target

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget classes:")
print(y.value_counts().sort_index())

## 8. Check class imbalance

In [ ]:
class_counts = y.value_counts().sort_index()
class_names = ["Genuine (0)", "Fraud (1)"]

plt.figure(figsize=(7, 4))
sns.barplot(x=class_names, y=[class_counts.get(0, 0), class_counts.get(1, 0)])
plt.title("Transaction Class Distribution")
plt.xlabel("Transaction class")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.show()

print("Class percentages:")
display((y.value_counts(normalize=True).sort_index() * 100).rename("Percentage"))

## 9. Split into training and testing sets

Stratification preserves the genuine/fraud class proportions in both sets. No resampling is performed here; imbalance can be handled during model training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())
print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

## 10. Scale `Time` and `Amount`

`V1`–`V28` are PCA-transformed features, so this notebook leaves them unchanged. The scaler is fitted only on `X_train`, then applied to both sets.

In [ ]:
columns_to_scale = ["Time", "Amount"]

missing_scale_columns = [col for col in columns_to_scale if col not in X_train.columns]
if missing_scale_columns:
    raise ValueError(f"Expected columns not found: {missing_scale_columns}")

scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train.loc[:, columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test.loc[:, columns_to_scale] = scaler.transform(X_test[columns_to_scale])

print("Scaling completed.")
display(X_train[columns_to_scale].describe())

## 11. Final preprocessing summary

In [ ]:
print("DATA PREPROCESSING COMPLETED")
print("-" * 40)
print(f"Rows after cleaning: {len(df):,}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Training samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")
print(f"Training fraud cases: {(y_train == 1).sum():,}")
print(f"Testing fraud cases: {(y_test == 1).sum():,}")

print("\nVariables ready for the next notebook:")
print("X_train, X_test, y_train, y_test")
print("\nNote: These variables are available in this notebook's kernel.")
print("A separate notebook has its own kernel; use a shared .py module or CSV files to pass data between notebooks.")

## Output variables for model training

After running all cells, the following variables are ready to use in this notebook:

- `X_train`: scaled training features
- `X_test`: scaled testing features
- `y_train`: training labels
- `y_test`: testing labels
- `scaler`: fitted `StandardScaler`

**Important:** Run the cells from top to bottom. This notebook does not create a `.pkl` file.